# tifpy — Verifikasi Tahap 1 TIF-SIG

**Spectral Information Graph (SIG) — prototipe komputasional**

Notebook ini menjalankan Tahap 1 roadmap TIF-SIG v2: verifikasi konsistensi
matematis konstruksi Hodge Laplacian dan substrat kompleks-kuaternion C-tensor-H.

Wawasan kunci: **C-tensor-H = M2(C)** — modul lokal kuaternion direpresentasikan
sebagai ruang C^2, koneksi SU(2) menjadi matriks 2x2 special-unitary.

## 1. Persiapan

In [1]:
import numpy as np
import tifpy as tp
print("tifpy siap. numpy", np.__version__)

tifpy siap. numpy 2.4.4


## 2. Bagian A - Hodge Laplacian & Teorema Hodge

Memverifikasi **dim ker(L_k) = beta_k** pada empat kompleks dengan topologi
yang diketahui. beta_k dihitung independen lewat rank operator boundary.

In [2]:
K_hollow = tp.SimplicialComplex.from_edges([0,1,2], [(0,1),(1,2),(0,2)])
K_filled = tp.SimplicialComplex.from_triangles([(0,1,2)])
K_sphere = tp.SimplicialComplex.from_triangles([(0,1,2),(0,1,3),(0,2,3),(1,2,3)])

def torus(m,n):
    vid = lambda r,c: (r%m)*n + (c%n)
    tris=[]
    for r in range(m):
        for c in range(n):
            a,b,cc,dd = vid(r,c),vid(r,c+1),vid(r+1,c),vid(r+1,c+1)
            tris += [(a,b,cc),(b,cc,dd)]
    return tp.SimplicialComplex.from_triangles(tris)
K_torus = torus(3,3)

for nama,K in [("Segitiga berlubang (S1)",K_hollow),
               ("Segitiga terisi",K_filled),
               ("Tetrahedron (S2)",K_sphere),
               ("Torus 3x3 (T2)",K_torus)]:
    betti  = tuple(tp.betti_number(K,k) for k in range(3))
    dimker = tuple(tp.kernel_dimension(tp.hodge_laplacian(K,k)) for k in range(3))
    ok = "OK" if betti==dimker else "GAGAL"
    print(f"{nama:28s}  Betti={betti}  dim ker={dimker}  [{ok}]")

Segitiga berlubang (S1)       Betti=(1, 1, 0)  dim ker=(1, 1, 0)  [OK]
Segitiga terisi               Betti=(1, 0, 0)  dim ker=(1, 0, 0)  [OK]
Tetrahedron (S2)              Betti=(1, 0, 1)  dim ker=(1, 0, 1)  [OK]
Torus 3x3 (T2)                Betti=(1, 2, 1)  dim ker=(1, 2, 1)  [OK]


## 3. Bagian B - Substrat C-tensor-H & Connection Laplacian

Setiap simpul membawa modul C^2. Koneksi SU(2) pada edge.
Memverifikasi self-adjoint, PSD, dan hubungan **holonomi <-> kernel**
(preview emergensi gauge, Konjektur K4).

In [3]:
rng = np.random.default_rng(42)
K = K_torus
beta0 = tp.betti_number(K,0)

conn_trivial = tp.trivial_connection(K)
conn_gauge   = tp.gauge_trivial_connection(K, rng)
conn_random  = tp.random_connection(K, rng)

for nama,conn in [("trivial",conn_trivial),
                   ("gauge-trivial",conn_gauge),
                   ("acak",conn_random)]:
    L  = tp.connection_laplacian(K, conn)
    sa = tp.is_self_adjoint(L)
    psd= tp.is_positive_semidefinite(L)
    dk = tp.kernel_dimension(L)
    H  = tp.holonomy(conn, [0,1,2])
    holo = np.linalg.norm(H - np.eye(2))
    print(f"Koneksi {nama:14s}  self-adj={sa}  PSD={psd}  "
          f"dim ker={dk}  ||holonomi-I||={holo:.3f}")

Koneksi trivial         self-adj=True  PSD=True  dim ker=2  ||holonomi-I||=0.000
Koneksi gauge-trivial   self-adj=True  PSD=True  dim ker=2  ||holonomi-I||=0.000
Koneksi acak            self-adj=True  PSD=True  dim ker=0  ||holonomi-I||=2.396


**Interpretasi:** koneksi trivial & gauge-trivial -> holonomi = I, dim ker = 2*beta0
(invariansi gauge). Koneksi acak -> holonomi != I, dim ker < 2*beta0:
holonomi non-trivial adalah muatan gauge yang terdeteksi (preview K4).

## 4. Suite verifikasi lengkap (32 uji)

In [4]:
import subprocess
hasil = subprocess.run(["python3","demo_step1.py"],capture_output=True,text=True)
print(hasil.stdout[-700:])

            dim ker = 2,  2*beta_0 = 2  --> tanpa loop, tiap koneksi gauge-trivial: holonomi mustahil

  RINGKASAN VERIFIKASI TAHAP 1

  Uji LULUS : 32 / 32

  >>> SELURUH UJI KONSISTENSI TAHAP 1 LULUS.
  >>> Konstruksi Hodge Laplacian pada SIG terbukti konsisten:
      - Operator boundary memenuhi d.d = 0
      - Hodge Laplacian self-adjoint dan positive semi-definite
      - Teorema Hodge terverifikasi: dim ker L_k = beta_k
      - Substrat C-tensor-H = M_2(C) terimplementasi konsisten
      - Holonomi SU(2) <-> kernel: preview mekanisme gauge (K4)


